<a href="https://colab.research.google.com/github/recursive-ai-dev/tensegrity-lm/blob/main/TensegrityLM_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TensegrityLM — Train, Grow, Retain, Export

A self-contained notebook for the TensegrityLM byte-level transformer
(RoPE + GQA, alternating local/global attention, SwiGLU/MoE FFN, homeostatic
residual-gain regularization, depth-scaled LayerDrop, and a confidence head).

**Pipeline:**

1. **Setup** — install dependencies, write `tensegrity_lm.py` and
   `tensegrity_nb_lib.py` to disk, import everything, pick a device.
2. **Data** — point at your own text (or let the notebook generate a small
   sample corpus) and prepare it for training.
3. **Model configuration** — define a `TensegrityConfig` and build the model.
4. **Train from scratch** — run the training loop with periodic evaluation
   and checkpointing (`ckpt_latest.pt` / `ckpt_best.pt`).
5. **Load a checkpoint and generate** — reload the model from disk and sample
   text from it.
6. **Grow the model** — increase capability via Net2Net-style,
   function-preserving growth (more layers and/or wider FFNs), and verify
   that the grown model computes *exactly* the same function as before growth.
7. **Continual training (retain old knowledge)** — train the grown model on
   new data while using Elastic Weight Consolidation (EWC) and replay of the
   old data to avoid catastrophic forgetting.
8. **Export to safetensors** — write portable weights + config, then reload
   and verify the roundtrip.

This notebook is fully self-contained: the two cells below write
`tensegrity_lm.py` (the model/training script) and `tensegrity_nb_lib.py`
(growth + continual-learning + export helpers) into the notebook's working
directory, then import from them. If you edit those two cells, **restart the
kernel** before re-running so the new versions are re-imported cleanly.

> **Tip:** the default configuration below is intentionally small so the
> whole notebook runs in a couple of minutes on a CPU. Scale up `n_layer`,
> `n_embd`, `block_size`, batch sizes, and step counts for real training runs
> (ideally on a GPU).


## 0. Dependencies

In [1]:
import importlib
import subprocess
import sys


def _ensure(pkg, import_name=None):
    """Import `import_name` (default: `pkg`), installing `pkg` via pip if needed.

    Tries a normal `pip install` first, then falls back to
    `--break-system-packages` for externally-managed Python environments
    (common on recent Debian/Ubuntu images).
    """
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
        return
    except ImportError:
        pass

    for extra_args in ([], ["--break-system-packages"]):
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", *extra_args, pkg],
                check=True,
            )
            importlib.import_module(import_name)
            return
        except Exception:
            continue

    raise RuntimeError(
        f"Could not install '{pkg}'. Please install it manually, e.g.:\n"
        f"    pip install {pkg}"
    )


for _pkg, _mod in [("torch", "torch"), ("numpy", "numpy"), ("safetensors", "safetensors")]:
    _ensure(_pkg, _mod)

print("dependencies OK")


dependencies OK


## 1. Write `tensegrity_lm.py`

The core model, configuration, tokenizer, data pipeline, checkpointing, and
training utilities. This is the same script used standalone from the command
line — the notebook just imports from it.


In [6]:
%%writefile tensegrity_lm.py
"""
TensegrityLM — a byte-level decoder-only language model.

Single-file, checkpointable, trainable implementation combining:
  - Rotary positional embeddings (RoPE)              [Geometry]
  - RMSNorm pre-norm transformer blocks              [Algebraic Math]
  - Grouped-Query Attention (GQA)                    [Algorithms]
  - Alternating local sliding-window / global causal attention [Space]
  - SwiGLU feed-forward, alternating dense / sparse Mixture-of-Experts
    with Switch-style load-balancing loss            [Social Dynamics]
  - Learnable per-layer residual gain + activation-RMS
    "homeostatic" set-point regularization           [Biology]
  - Depth-scaled stochastic layer drop (LayerDrop)   [Evolution]
  - Cosine LR schedule with warmup ("temperature annealing") [Physics]
  - Optional confidence head for adaptive sampling temperature
                                                      [Psychology]
  - Byte-level tokenizer, zero external deps         [English Language]

Honesty note: every individual mechanism above is established in the
literature (RoPE, GQA, SwiGLU, MoE+load-balancing, RMSNorm, LayerDrop,
LayerScale-style gains, cosine schedules). The specific combination —
particularly the homeostatic residual-gain regularization framed as an
explicit activation-RMS set-point loss, paired with depth-scaled layer
drop and alternating local/global GQA — is a coherent but UNVALIDATED
recombination. Train it, measure it, don't assume it beats a vanilla
transformer of equal size until you've run the comparison yourself.

Usage:
    python tensegrity_lm.py prepare --data_path mydata.txt --out_dir data/mycorpus
    python tensegrity_lm.py train    --data_dir data/mycorpus --out_dir runs/run1
    python tensegrity_lm.py train    --data_dir data/mycorpus --out_dir runs/run1 --init_from resume
    python tensegrity_lm.py generate --ckpt runs/run1/ckpt_best.pt --prompt "Hello" --max_new_tokens 200
    python tensegrity_lm.py eval     --ckpt runs/run1/ckpt_best.pt --data_dir data/mycorpus
"""

import argparse
import dataclasses
import glob
import json
import math
import os
import sys
import time
from dataclasses import dataclass, field, asdict
from typing import Optional, Tuple, List, Dict, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# =============================================================================
# Tokenizer
# =============================================================================

class ByteTokenizer:
    """
    Byte-level tokenizer. Vocabulary = 256 raw byte values + 3 special tokens:
        256 = BOS (beginning of sequence)
        257 = EOS (end of sequence / document separator)
        258 = PAD (padding, unused by default but reserved)

    Zero training required, lossless for any UTF-8 (or arbitrary binary) input.
    """
    VOCAB_SIZE = 259
    BOS = 256
    EOS = 257
    PAD = 258

    def encode(self, text: str) -> List[int]:
        return list(text.encode("utf-8", errors="replace"))

    def encode_bytes(self, data: bytes) -> List[int]:
        return list(data)

    def decode(self, ids: List[int]) -> str:
        byte_vals = bytes(i for i in ids if 0 <= i <= 255)
        return byte_vals.decode("utf-8", errors="replace")


# =============================================================================
# Configuration
# =============================================================================

@dataclass
class TensegrityConfig:
    # Core sizes
    vocab_size: int = ByteTokenizer.VOCAB_SIZE
    block_size: int = 256          # max sequence length / context window
    n_layer: int = 6
    n_embd: int = 384
    n_head: int = 6                # query heads
    n_kv_head: int = 2              # key/value heads (n_head % n_kv_head == 0) -> GQA
    dropout: float = 0.0

    # Attention pattern (local/global alternation -> "Space" mapping)
    sliding_window: int = 128       # local attention window size
    global_every: int = 3           # every Nth layer (1-indexed) uses full causal attention

    # Feed-forward / MoE
    ffn_mult: float = 8.0 / 3.0     # SwiGLU hidden-dim multiplier (matches ~4x ReLU param count)
    ffn_align: int = 64             # round FFN hidden dim up to multiple of this
    moe_every: int = 2              # every Nth layer (1-indexed) uses MoE FFN instead of dense
    n_experts: int = 4
    moe_top_k: int = 2
    moe_aux_weight: float = 0.01    # load-balancing auxiliary loss weight

    # Homeostatic residual gain regularization ("Biology")
    homeo_target: float = 1.0       # target activation RMS for each sublayer output
    homeo_weight: float = 0.01      # weight of homeostasis loss term
    gain_init: Optional[float] = None  # if None, uses 1/sqrt(2*n_layer) (DeepNet-style)

    # Stochastic depth ("Evolution") — linear schedule, 0 at layer 1 -> layerdrop_max at layer n_layer
    layerdrop_max: float = 0.1

    # Confidence head ("Psychology") — predicts P(token correct) for adaptive sampling temperature
    use_confidence_head: bool = True
    confidence_weight: float = 0.05

    # Misc
    bias: bool = False               # use bias in Linear layers
    init_std: float = 0.02
    grad_checkpoint: bool = False    # gradient checkpointing for memory efficiency (training only)

    def __post_init__(self):
        assert self.n_embd % self.n_head == 0, "n_embd must be divisible by n_head"
        head_dim = self.n_embd // self.n_head
        assert head_dim % 2 == 0, "head_dim (n_embd // n_head) must be even for RoPE"
        assert self.n_head % self.n_kv_head == 0, "n_head must be divisible by n_kv_head for GQA"
        assert self.n_experts >= 1, "n_experts must be >= 1"
        assert 1 <= self.moe_top_k <= self.n_experts, "moe_top_k must be in [1, n_experts]"
        assert self.global_every >= 1
        assert self.moe_every >= 1
        assert 0.0 <= self.layerdrop_max < 1.0
        assert self.sliding_window >= 1
        if self.gain_init is None:
            self.gain_init = 1.0 / math.sqrt(2.0 * self.n_layer)


# =============================================================================
# Core building blocks
# =============================================================================

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Compute norm in fp32 for stability regardless of input dtype (e.g. under autocast)
        in_dtype = x.dtype
        x_fp32 = x.float()
        rms = torch.rsqrt(x_fp32.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        out = (x_fp32 * rms).to(in_dtype)
        return out * self.weight


def precompute_rope(head_dim: int, max_seq_len: int, base: float = 10000.0,
                     device=None, dtype=torch.float32) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Precompute cos/sin tables for RoPE. Returns tensors of shape (max_seq_len, head_dim // 2).
    """
    assert head_dim % 2 == 0
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device, dtype=torch.float32) / head_dim))
    t = torch.arange(max_seq_len, device=device, dtype=torch.float32)
    freqs = torch.outer(t, inv_freq)  # (max_seq_len, head_dim // 2)
    cos = freqs.cos().to(dtype)
    sin = freqs.sin().to(dtype)
    return cos, sin


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    Apply rotary position embedding to x.
    x:   (B, H, T, D)  where D is the head dimension (even)
    cos: (T, D // 2)
    sin: (T, D // 2)
    """
    B, H, T, D = x.shape
    half = D // 2
    x1 = x[..., :half]
    x2 = x[..., half:]
    cos_ = cos[:T].view(1, 1, T, half)
    sin_ = sin[:T].view(1, 1, T, half)
    # Standard "rotate half" RoPE formulation
    out1 = x1 * cos_ - x2 * sin_
    out2 = x2 * cos_ + x1 * sin_
    return torch.cat([out1, out2], dim=-1)


def activation_rms(x: torch.Tensor) -> torch.Tensor:
    """Mean RMS of activations across the feature dimension, averaged over batch/tokens."""
    return x.float().pow(2).mean(dim=-1).sqrt().mean()


# =============================================================================
# Attention (Grouped-Query, with alternating local/global masks)
# =============================================================================

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: TensegrityConfig, is_global: bool):
        super().__init__()
        self.n_head = cfg.n_head
        self.n_kv_head = cfg.n_kv_head
        self.head_dim = cfg.n_embd // cfg.n_head
        self.n_rep = cfg.n_head // cfg.n_kv_head
        self.is_global = is_global
        self.sliding_window = cfg.sliding_window
        self.dropout_p = cfg.dropout

        self.q_proj = nn.Linear(cfg.n_embd, cfg.n_head * self.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.n_embd, cfg.n_kv_head * self.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.n_embd, cfg.n_kv_head * self.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * self.head_dim, cfg.n_embd, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor,
                attn_mask: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)       # (B, nH, T, D)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)    # (B, nKV, T, D)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)    # (B, nKV, T, D)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)  # (B, nH, T, D)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # attn_mask: (T, T) bool, True = keep / attend, False = masked out
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=False,  # causality is encoded directly in attn_mask
        )
        y = y.transpose(1, 2).contiguous().view(B, T, self.n_head * self.head_dim)
        y = self.o_proj(y)
        y = self.resid_dropout(y)
        return y


def build_attention_masks(block_size: int, sliding_window: int, device=None) -> Dict[str, torch.Tensor]:
    """
    Build boolean attention masks of shape (block_size, block_size).
    True = positions that may attend to each other (query i attends to key j).
    """
    idx = torch.arange(block_size, device=device)
    i = idx.view(-1, 1)
    j = idx.view(1, -1)
    causal = j <= i
    local = causal & (j > (i - sliding_window))
    return {"global": causal, "local": local}


# =============================================================================
# Feed-forward: SwiGLU (dense) and Mixture-of-Experts
# =============================================================================

def ffn_hidden_dim(n_embd: int, mult: float, align: int) -> int:
    hidden = int(n_embd * mult)
    hidden = align * ((hidden + align - 1) // align)
    return max(hidden, align)


class SwiGLU(nn.Module):
    def __init__(self, n_embd: int, hidden_dim: int, bias: bool, dropout: float):
        super().__init__()
        self.w_gate = nn.Linear(n_embd, hidden_dim, bias=bias)
        self.w_up = nn.Linear(n_embd, hidden_dim, bias=bias)
        self.w_down = nn.Linear(hidden_dim, n_embd, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))


class MoEFeedForward(nn.Module):
    """
    Top-k routed Mixture of Experts with SwiGLU experts and a Switch-Transformer
    style load-balancing auxiliary loss.
    """

    def __init__(self, cfg: TensegrityConfig, hidden_dim: int):
        super().__init__()
        self.n_experts = cfg.n_experts
        self.top_k = cfg.moe_top_k
        self.n_embd = cfg.n_embd
        self.aux_weight = cfg.moe_aux_weight

        self.router = nn.Linear(cfg.n_embd, cfg.n_experts, bias=False)
        self.experts = nn.ModuleList([
            SwiGLU(cfg.n_embd, hidden_dim, cfg.bias, cfg.dropout) for _ in range(cfg.n_experts)
        ])

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, C = x.shape
        x_flat = x.reshape(-1, C)  # (N, C), N = B*T
        N = x_flat.shape[0]

        # Routing: compute in fp32 for numerical stability
        router_logits = self.router(x_flat).float()           # (N, E)
        router_probs = F.softmax(router_logits, dim=-1)        # (N, E)

        topk_probs, topk_idx = torch.topk(router_probs, self.top_k, dim=-1)  # (N, K)
        # Re-normalize the top-k probabilities so they sum to 1 per token
        topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True).clamp_min(1e-9)

        out = torch.zeros_like(x_flat)

        for e in range(self.n_experts):
            # tokens where expert e is among the chosen top-k
            expert_mask = (topk_idx == e)              # (N, K) bool
            if not expert_mask.any():
                continue
            token_idx, slot_idx = expert_mask.nonzero(as_tuple=True)  # indices into N and K
            if token_idx.numel() == 0:
                continue
            tokens_in = x_flat.index_select(0, token_idx)             # (M, C)
            expert_out = self.experts[e](tokens_in)                    # (M, C)
            weights = topk_probs[token_idx, slot_idx].unsqueeze(-1).to(expert_out.dtype)  # (M, 1)
            weighted = (expert_out * weights).to(out.dtype)
            out.index_add_(0, token_idx, weighted)

        out = out.reshape(B, T, C)

        # --- Load-balancing auxiliary loss (Switch Transformer formulation) ---
        # f_e = fraction of tokens for which expert e is in the top-1 (hard assignment proxy)
        # P_e = mean router probability assigned to expert e (soft, differentiable)
        with torch.no_grad():
            top1_idx = topk_idx[:, 0]  # (N,)
            one_hot = F.one_hot(top1_idx, num_classes=self.n_experts).float()  # (N, E)
            f = one_hot.mean(dim=0)  # (E,)
        P = router_probs.mean(dim=0)  # (E,) — carries gradient
        aux_loss = self.aux_weight * self.n_experts * (f * P).sum()

        return out, aux_loss


# =============================================================================
# Transformer block
# =============================================================================

class Block(nn.Module):
    def __init__(self, cfg: TensegrityConfig, layer_idx: int):
        super().__init__()
        self.layer_idx = layer_idx
        is_global = ((layer_idx + 1) % cfg.global_every == 0)
        is_moe = ((layer_idx + 1) % cfg.moe_every == 0)
        self.is_moe = is_moe

        self.ln1 = RMSNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg, is_global=is_global)
        self.ln2 = RMSNorm(cfg.n_embd)

        hidden_dim = ffn_hidden_dim(cfg.n_embd, cfg.ffn_mult, cfg.ffn_align)
        if is_moe:
            self.ffn = MoEFeedForward(cfg, hidden_dim)
        else:
            self.ffn = SwiGLU(cfg.n_embd, hidden_dim, cfg.bias, cfg.dropout)

        # Homeostatic residual gains (learnable per-layer, per-sublayer scalars)
        self.attn_gain = nn.Parameter(torch.tensor(float(cfg.gain_init)))
        self.ffn_gain = nn.Parameter(torch.tensor(float(cfg.gain_init)))
        self.homeo_target = cfg.homeo_target

        # Depth-scaled stochastic depth probability
        n_layer = cfg.n_layer
        if n_layer > 1:
            self.drop_p = cfg.layerdrop_max * (layer_idx / (n_layer - 1))
        else:
            self.drop_p = 0.0

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor,
                attn_mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # Stochastic depth: during training, randomly skip the entire block (batch-level)
        if self.training and self.drop_p > 0.0:
            if torch.rand((), device=x.device).item() < self.drop_p:
                zero = x.new_zeros(())
                return x, zero, zero

        attn_out = self.attn(self.ln1(x), cos, sin, attn_mask)
        homeo_attn = (activation_rms(attn_out) - self.homeo_target).pow(2)
        x = x + self.attn_gain * attn_out

        if self.is_moe:
            ffn_out, moe_aux = self.ffn(self.ln2(x))
        else:
            ffn_out = self.ffn(self.ln2(x))
            moe_aux = x.new_zeros(())
        homeo_ffn = (activation_rms(ffn_out) - self.homeo_target).pow(2)
        x = x + self.ffn_gain * ffn_out

        homeo = homeo_attn + homeo_ffn
        return x, homeo, moe_aux


# =============================================================================
# Full model
# =============================================================================

class TensegrityLM(nn.Module):
    def __init__(self, cfg: TensegrityConfig):
        super().__init__()
        self.cfg = cfg
        self.grad_checkpoint = cfg.grad_checkpoint

        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg, i) for i in range(cfg.n_layer)])
        self.ln_f = RMSNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

        # Weight tying
        self.token_emb.weight = self.lm_head.weight

        # Optional confidence head: predicts P(argmax prediction == target)
        if cfg.use_confidence_head:
            self.confidence_head = nn.Linear(cfg.n_embd, 1, bias=True)
        else:
            self.confidence_head = None

        head_dim = cfg.n_embd // cfg.n_head
        cos, sin = precompute_rope(head_dim, cfg.block_size)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        masks = build_attention_masks(cfg.block_size, cfg.sliding_window)
        self.register_buffer("mask_global", masks["global"], persistent=False)
        self.register_buffer("mask_local", masks["local"], persistent=False)

        self.apply(self._init_weights)

        n_moe = sum(1 for b in self.blocks if b.is_moe)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"[TensegrityLM] layers={cfg.n_layer} (moe_layers={n_moe}) "
              f"n_embd={cfg.n_embd} n_head={cfg.n_head} n_kv_head={cfg.n_kv_head} "
              f"vocab={cfg.vocab_size} block_size={cfg.block_size}")
        print(f"[TensegrityLM] total parameters: {n_params:,} ({n_params/1e6:.2f}M)")

    def _init_weights(self, module: nn.Module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=self.cfg.init_std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=self.cfg.init_std)

    def get_num_params(self) -> int:
        return sum(p.numel() for p in self.parameters())

    def forward(self, idx: torch.Tensor, targets: Optional[torch.Tensor] = None
                 ) -> Tuple[torch.Tensor, Optional[Dict[str, torch.Tensor]], Optional[torch.Tensor]]:
        """
        idx:     (B, T) integer token ids, T <= block_size
        targets: (B, T) integer token ids (next-token targets), or None for inference

        Returns:
            logits: (B, T, vocab_size)
            losses: dict of scalar tensors (only if targets is not None), else None
            confidence: (B, T) sigmoid confidence scores (only if targets is None and head enabled), else None
        """
        B, T = idx.shape
        assert T <= self.cfg.block_size, f"sequence length {T} exceeds block_size {self.cfg.block_size}"

        x = self.token_emb(idx)  # (B, T, C)
        x = self.drop(x)

        cos = self.rope_cos[:T]
        sin = self.rope_sin[:T]
        mask_global = self.mask_global[:T, :T]
        mask_local = self.mask_local[:T, :T]

        total_homeo = x.new_zeros(())
        total_moe_aux = x.new_zeros(())
        n_active = 0

        use_checkpoint = self.grad_checkpoint and self.training and x.requires_grad

        for block in self.blocks:
            attn_mask = mask_global if block.attn.is_global else mask_local
            if use_checkpoint:
                x, homeo, moe_aux = torch.utils.checkpoint.checkpoint(
                    block, x, cos, sin, attn_mask, use_reentrant=False
                )
            else:
                x, homeo, moe_aux = block(x, cos, sin, attn_mask)
            total_homeo = total_homeo + homeo
            total_moe_aux = total_moe_aux + moe_aux
            n_active += 1

        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is not None:
            ce_loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1),
                ignore_index=-1,
            )

            denom = max(1, n_active * 2)
            avg_homeo = total_homeo / denom

            n_moe_layers = sum(1 for b in self.blocks if b.is_moe)
            avg_moe_aux = total_moe_aux / max(1, n_moe_layers)

            losses = {
                "ce_loss": ce_loss,
                "homeo_loss": avg_homeo,
                "moe_aux_loss": avg_moe_aux,
            }

            total_loss = ce_loss + self.cfg.homeo_weight * avg_homeo + avg_moe_aux

            if self.confidence_head is not None:
                conf_logits = self.confidence_head(x).squeeze(-1)  # (B, T)
                with torch.no_grad():
                    preds = logits.argmax(dim=-1)  # (B, T)
                    correct = (preds == targets).float()
                    valid = (targets != -1).float()
                conf_loss_raw = F.binary_cross_entropy_with_logits(
                    conf_logits, correct, reduction="none"
                )
                denom_c = valid.sum().clamp_min(1.0)
                conf_loss = (conf_loss_raw * valid).sum() / denom_c
                losses["confidence_loss"] = conf_loss
                total_loss = total_loss + self.cfg.confidence_weight * conf_loss

            losses["total_loss"] = total_loss
            return logits, losses, None

        else:
            confidence = None
            if self.confidence_head is not None:
                conf_logits = self.confidence_head(x).squeeze(-1)  # (B, T)
                confidence = torch.sigmoid(conf_logits)
            return logits, None, confidence

    # -------------------------------------------------------------------
    # Generation
    # -------------------------------------------------------------------
    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int,
                  temperature: float = 1.0, top_k: Optional[int] = None,
                  top_p: Optional[float] = None,
                  use_confidence_temp: bool = False,
                  conf_temp_min: float = 0.5, conf_temp_max: float = 1.5,
                  stop_on_eos: bool = True) -> torch.Tensor:
        """
        idx: (B, T) starting context.
        Returns (B, T + max_new_tokens) — may be shorter if EOS is hit and stop_on_eos
        is True (in which case generation stops for the whole batch at that step).
        """
        self.eval()
        for _ in range(max_new_tokens):
            T = idx.size(1)
            idx_cond = idx if T <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
            logits, _, confidence = self.forward(idx_cond, targets=None)
            logits = logits[:, -1, :]  # (B, V)

            eff_temp = temperature
            if use_confidence_temp and confidence is not None:
                conf = confidence[:, -1]  # (B,)
                # low confidence -> higher temperature (more exploration)
                scale = conf_temp_max - conf * (conf_temp_max - conf_temp_min)
                eff_temp = temperature * scale.mean().item()
                eff_temp = max(eff_temp, 1e-4)

            logits = logits / eff_temp

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")

            if top_p is not None:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
                probs = F.softmax(sorted_logits, dim=-1)
                cumprobs = torch.cumsum(probs, dim=-1)
                cutoff = cumprobs > top_p
                cutoff[:, 1:] = cutoff[:, :-1].clone()
                cutoff[:, 0] = False
                sorted_logits[cutoff] = -float("inf")
                logits = torch.full_like(logits, -float("inf")).scatter(1, sorted_idx, sorted_logits)

            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat([idx, next_tok], dim=1)

            if stop_on_eos and (next_tok == ByteTokenizer.EOS).all():
                break

        return idx

    # -------------------------------------------------------------------
    # Optimizer configuration
    # -------------------------------------------------------------------
    def configure_optimizer(self, weight_decay: float, learning_rate: float,
                             betas: Tuple[float, float], device_type: str) -> torch.optim.Optimizer:
        decay_params = []
        no_decay_params = []
        seen = set()
        for name, p in self.named_parameters():
            if not p.requires_grad:
                continue
            if id(p) in seen:
                continue
            seen.add(id(p))
            if p.dim() >= 2:
                decay_params.append(p)
            else:
                no_decay_params.append(p)

        groups = [
            {"params": decay_params, "weight_decay": weight_decay},
            {"params": no_decay_params, "weight_decay": 0.0},
        ]
        use_fused = (device_type == "cuda") and ("fused" in torch.optim.AdamW.__init__.__code__.co_varnames)
        extra = {"fused": True} if use_fused else {}
        optimizer = torch.optim.AdamW(groups, lr=learning_rate, betas=betas, **extra)
        return optimizer


# =============================================================================
# Data pipeline
# =============================================================================

def prepare_data(data_path: str, out_dir: str, val_fraction: float = 0.1, seed: int = 1337):
    """
    Read a text file or directory of files, byte-encode (with EOS separators between
    documents), split into train/val, and write uint16 .bin files for memmap access.
    """
    os.makedirs(out_dir, exist_ok=True)
    tok = ByteTokenizer()

    if os.path.isdir(data_path):
        files = sorted(glob.glob(os.path.join(data_path, "**", "*"), recursive=True))
        files = [f for f in files if os.path.isfile(f)]
    else:
        files = [data_path]

    if not files:
        raise ValueError(f"No files found at {data_path}")

    all_ids: List[int] = []
    for f in files:
        with open(f, "rb") as fh:
            raw = fh.read()
        all_ids.extend(tok.encode_bytes(raw))
        all_ids.append(ByteTokenizer.EOS)

    arr = np.array(all_ids, dtype=np.uint16)
    n = len(arr)
    n_val = max(1, int(n * val_fraction))
    n_train = n - n_val
    if n_train <= 0:
        raise ValueError("Dataset too small after train/val split; provide more data.")

    train_arr = arr[:n_train]
    val_arr = arr[n_train:]

    train_path = os.path.join(out_dir, "train.bin")
    val_path = os.path.join(out_dir, "val.bin")
    train_arr.tofile(train_path)
    val_arr.tofile(val_path)

    meta = {
        "vocab_size": ByteTokenizer.VOCAB_SIZE,
        "n_train_tokens": int(n_train),
        "n_val_tokens": int(n_val),
        "source_files": files,
    }
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    print(f"[prepare] wrote {n_train:,} train tokens -> {train_path}")
    print(f"[prepare] wrote {n_val:,} val tokens -> {val_path}")


def get_batch(data_dir: str, split: str, batch_size: int, block_size: int,
               device: str) -> Tuple[torch.Tensor, torch.Tensor]:
    path = os.path.join(data_dir, f"{split}.bin")
    arr = np.memmap(path, dtype=np.uint16, mode="r")
    n = len(arr)
    if n <= block_size + 1:
        raise ValueError(
            f"{split}.bin has only {n} tokens, need > block_size+1 ({block_size + 1}). "
            f"Use a larger corpus or smaller --block_size."
        )
    max_start = n - block_size - 1
    starts = np.random.randint(0, max_start, size=(batch_size,))
    x = np.stack([arr[s:s + block_size].astype(np.int64) for s in starts])
    y = np.stack([arr[s + 1:s + 1 + block_size].astype(np.int64) for s in starts])
    x_t = torch.from_numpy(x)
    y_t = torch.from_numpy(y)
    if device == "cuda":
        x_t = x_t.pin_memory().to(device, non_blocking=True)
        y_t = y_t.pin_memory().to(device, non_blocking=True)
    else:
        x_t = x_t.to(device)
        y_t = y_t.to(device)
    return x_t, y_t


# =============================================================================
# Checkpointing
# =============================================================================

def save_checkpoint(path: str, model: nn.Module, optimizer: torch.optim.Optimizer,
                     cfg: TensegrityConfig, step: int, best_val_loss: float,
                     rng_state: Dict[str, Any]):
    raw_model = model.module if hasattr(model, "module") else model
    ckpt = {
        "model_state_dict": raw_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": asdict(cfg),
        "step": step,
        "best_val_loss": best_val_loss,
        "rng_state": rng_state,
    }
    tmp_path = path + ".tmp"
    torch.save(ckpt, tmp_path)
    os.replace(tmp_path, path)


def load_checkpoint(path: str, device: str) -> Dict[str, Any]:
    ckpt = torch.load(path, map_location=device, weights_only=False)
    return ckpt


def capture_rng_state() -> Dict[str, Any]:
    state = {
        "torch": torch.get_rng_state(),
        "numpy": np.random.get_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def restore_rng_state(state: Dict[str, Any]):
    if state is None:
        return
    if "torch" in state:
        torch.set_rng_state(state["torch"])
    if "numpy" in state:
        np.random.set_state(state["numpy"])
    if "cuda" in state and torch.cuda.is_available():
        try:
            torch.cuda.set_rng_state_all(state["cuda"])
        except Exception:
            pass


# =============================================================================
# Learning rate schedule (cosine with warmup — "temperature annealing")
# =============================================================================

def get_lr(step: int, warmup_steps: int, max_steps: int, max_lr: float, min_lr: float) -> float:
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= max_steps:
        return min_lr
    decay_ratio = (step - warmup_steps) / max(1, (max_steps - warmup_steps))
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)


# =============================================================================
# Device / dtype helpers
# =============================================================================

def pick_device(requested: str = "auto") -> str:
    if requested != "auto":
        return requested
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def pick_dtype(requested: str, device: str) -> torch.dtype:
    if requested != "auto":
        return {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[requested]
    if device == "cuda" and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    if device == "cuda":
        return torch.float16
    return torch.float32


# =============================================================================
# Training
# =============================================================================

def build_config_from_args(args) -> TensegrityConfig:
    cfg_kwargs = {}
    for f in dataclasses.fields(TensegrityConfig):
        if hasattr(args, f.name):
            val = getattr(args, f.name)
            if val is not None:
                cfg_kwargs[f.name] = val
    return TensegrityConfig(**cfg_kwargs)


def cmd_train(args):
    is_ddp = int(os.environ.get("RANK", -1)) != -1
    if is_ddp:
        import torch.distributed as dist
        from torch.nn.parallel import DistributedDataParallel as DDP
        dist.init_process_group(backend="nccl")
        ddp_rank = int(os.environ["RANK"])
        ddp_local_rank = int(os.environ["LOCAL_RANK"])
        ddp_world_size = int(os.environ["WORLD_SIZE"])
        device = f"cuda:{ddp_local_rank}"
        torch.cuda.set_device(device)
        master_process = (ddp_rank == 0)
        seed_offset = ddp_rank
    else:
        ddp_world_size = 1
        master_process = True
        seed_offset = 0
        device = pick_device(args.device)

    torch.manual_seed(args.seed + seed_offset)
    np.random.seed(args.seed + seed_offset)

    os.makedirs(args.out_dir, exist_ok=True)

    meta_path = os.path.join(args.data_dir, "meta.json")
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        if master_process:
            print(f"[data] {meta['n_train_tokens']:,} train tokens, "
                  f"{meta['n_val_tokens']:,} val tokens, vocab={meta['vocab_size']}")

    device_type = "cuda" if "cuda" in device else ("mps" if device == "mps" else "cpu")
    dtype = pick_dtype(args.dtype, device_type)
    if master_process:
        print(f"[train] device={device} dtype={dtype}")

    ckpt_resume = None
    if args.init_from == "resume":
        latest_path = os.path.join(args.out_dir, "ckpt_latest.pt")
        if os.path.exists(latest_path):
            ckpt_resume = load_checkpoint(latest_path, device)
            if master_process:
                print(f"[resume] loaded checkpoint from {latest_path} at step {ckpt_resume['step']}")
        else:
            if master_process:
                print(f"[resume] no checkpoint found at {latest_path}, starting from scratch")

    if ckpt_resume is not None:
        cfg = TensegrityConfig(**ckpt_resume["config"])
    else:
        cfg = build_config_from_args(args)

    model = TensegrityLM(cfg)
    model.to(device)

    if ckpt_resume is not None:
        model.load_state_dict(ckpt_resume["model_state_dict"])

    optimizer = model.configure_optimizer(
        weight_decay=args.weight_decay, learning_rate=args.learning_rate,
        betas=(args.beta1, args.beta2), device_type=device_type,
    )
    if ckpt_resume is not None:
        optimizer.load_state_dict(ckpt_resume["optimizer_state_dict"])
        restore_rng_state(ckpt_resume.get("rng_state"))

    start_step = ckpt_resume["step"] if ckpt_resume is not None else 0
    best_val_loss = ckpt_resume["best_val_loss"] if ckpt_resume is not None else float("inf")

    raw_model = model
    if args.compile:
        try:
            model = torch.compile(model)
            if master_process:
                print("[train] torch.compile enabled")
        except Exception as e:
            if master_process:
                print(f"[train] torch.compile failed ({e}), continuing without it")

    if is_ddp:
        model = DDP(model, device_ids=[ddp_local_rank], find_unused_parameters=True)
        raw_model = model.module

    use_amp = dtype in (torch.float16, torch.bfloat16) and device_type in ("cuda", "cpu")
    amp_dtype = dtype if use_amp else torch.float32
    scaler = torch.amp.GradScaler(enabled=(dtype == torch.float16 and device_type == "cuda"))

    log_path = os.path.join(args.out_dir, "train_log.jsonl")
    log_file = open(log_path, "a") if master_process else None

    t0 = time.time()
    step = start_step
    running_loss = 0.0

    model.train()
    optimizer.zero_grad(set_to_none=True)

    while step < args.max_steps:
        lr = get_lr(step, args.warmup_steps, args.max_steps, args.learning_rate, args.min_lr)
        for group in optimizer.param_groups:
            group["lr"] = lr

        for micro_step in range(args.grad_accum_steps):
            x, y = get_batch(args.data_dir, "train", args.batch_size, cfg.block_size, device)

            if is_ddp:
                model.require_backward_grad_sync = (micro_step == args.grad_accum_steps - 1)

            with torch.amp.autocast(device_type=device_type if device_type != "mps" else "cpu",
                                      dtype=amp_dtype, enabled=use_amp):
                logits, losses, _ = model(x, y)
                loss = losses["total_loss"] / args.grad_accum_steps

            if scaler.is_enabled():
                scaler.scale(loss).backward()
            else:
                loss.backward()

            running_loss += losses["total_loss"].item() / args.grad_accum_steps

        if args.grad_clip > 0.0:
            if scaler.is_enabled():
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)

        if scaler.is_enabled():
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        if step % args.log_interval == 0 and master_process:
            dt = time.time() - t0
            avg_loss = running_loss / max(1, args.log_interval if step > start_step else 1)
            ce = losses["ce_loss"].item()
            homeo = losses["homeo_loss"].item()
            moe_aux = losses["moe_aux_loss"].item()
            conf = losses.get("confidence_loss")
            conf_str = f" conf={conf.item():.4f}" if conf is not None else ""
            print(f"step {step:6d} | lr {lr:.2e} | total {losses['total_loss'].item():.4f} "
                  f"(ce {ce:.4f} homeo {homeo:.4f} moe {moe_aux:.4f}{conf_str}) | {dt:.1f}s")
            log_file.write(json.dumps({
                "step": step, "lr": lr, "total_loss": losses["total_loss"].item(),
                "ce_loss": ce, "homeo_loss": homeo, "moe_aux_loss": moe_aux,
                "time": dt,
            }) + "\n")
            log_file.flush()
            running_loss = 0.0

        if step > 0 and step % args.eval_interval == 0 and master_process:
            val_loss = evaluate(raw_model, args.data_dir, cfg, device, device_type, amp_dtype,
                                  use_amp, args.eval_iters)
            print(f"step {step:6d} | val_loss {val_loss:.4f}")
            log_file.write(json.dumps({"step": step, "val_loss": val_loss}) + "\n")
            log_file.flush()

            rng_state = capture_rng_state()
            save_checkpoint(os.path.join(args.out_dir, "ckpt_latest.pt"), raw_model, optimizer,
                             cfg, step, best_val_loss, rng_state)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                save_checkpoint(os.path.join(args.out_dir, "ckpt_best.pt"), raw_model, optimizer,
                                 cfg, step, best_val_loss, rng_state)
                print(f"step {step:6d} | new best val_loss {best_val_loss:.4f}, checkpoint saved")

        step += 1

    if master_process:
        val_loss = evaluate(raw_model, args.data_dir, cfg, device, device_type, amp_dtype,
                              use_amp, args.eval_iters)
        rng_state = capture_rng_state()
        save_checkpoint(os.path.join(args.out_dir, "ckpt_latest.pt"), raw_model, optimizer,
                         cfg, step, best_val_loss, rng_state)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(os.path.join(args.out_dir, "ckpt_best.pt"), raw_model, optimizer,
                             cfg, step, best_val_loss, rng_state)
        print(f"[train] done. final val_loss={val_loss:.4f} best_val_loss={best_val_loss:.4f}")
        log_file.close()

    if is_ddp:
        import torch.distributed as dist
        dist.destroy_process_group()


@torch.no_grad()
def evaluate(model: nn.Module, data_dir: str, cfg: TensegrityConfig, device: str,
             device_type: str, amp_dtype: torch.dtype, use_amp: bool, eval_iters: int) -> float:
    model.eval()
    losses = []
    for split in ["val"]:
        for _ in range(eval_iters):
            x, y = get_batch(data_dir, split, batch_size=8, block_size=cfg.block_size, device=device)
            with torch.amp.autocast(device_type=device_type if device_type != "mps" else "cpu",
                                      dtype=amp_dtype, enabled=use_amp):
                _, loss_dict, _ = model(x, y)
            losses.append(loss_dict["ce_loss"].item())
    model.train()
    return float(np.mean(losses))


# =============================================================================
# Generation / Eval CLI commands
# =============================================================================

def cmd_generate(args):
    device = pick_device(args.device)
    ckpt = load_checkpoint(args.ckpt, device)
    cfg = TensegrityConfig(**ckpt["config"])
    model = TensegrityLM(cfg)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()

    tok = ByteTokenizer()
    if args.seed is not None:
        torch.manual_seed(args.seed)

    if args.prompt:
        ids = tok.encode(args.prompt)
    else:
        ids = []
    if args.add_bos:
        ids = [ByteTokenizer.BOS] + ids
    if not ids:
        ids = [ByteTokenizer.BOS]

    idx = torch.tensor([ids], dtype=torch.long, device=device)

    out = model.generate(
        idx, max_new_tokens=args.max_new_tokens, temperature=args.temperature,
        top_k=args.top_k, top_p=args.top_p,
        use_confidence_temp=args.use_confidence_temp,
        stop_on_eos=not args.no_stop_on_eos,
    )

    out_ids = out[0].tolist()
    text = tok.decode(out_ids)
    print(text)


def cmd_eval(args):
    device = pick_device(args.device)
    ckpt = load_checkpoint(args.ckpt, device)
    cfg = TensegrityConfig(**ckpt["config"])
    model = TensegrityLM(cfg)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()

    device_type = "cuda" if "cuda" in device else ("mps" if device == "mps" else "cpu")
    dtype = pick_dtype(args.dtype, device_type)
    use_amp = dtype in (torch.float16, torch.bfloat16) and device_type in ("cuda", "cpu")

    val_loss = evaluate(model, args.data_dir, cfg, device, device_type, dtype, use_amp, args.eval_iters)
    print(f"val_loss = {val_loss:.4f}")
    print(f"val_perplexity = {math.exp(val_loss):.4f}")


# =============================================================================
# Argument parsing
# =============================================================================

def add_model_args(p: argparse.ArgumentParser):
    p.add_argument("--block_size", type=int, default=None)
    p.add_argument("--n_layer", type=int, default=None)
    p.add_argument("--n_embd", type=int, default=None)
    p.add_argument("--n_head", type=int, default=None)
    p.add_argument("--n_kv_head", type=int, default=None)
    p.add_argument("--dropout", type=float, default=None)
    p.add_argument("--sliding_window", type=int, default=None)
    p.add_argument("--global_every", type=int, default=None)
    p.add_argument("--ffn_mult", type=float, default=None)
    p.add_argument("--ffn_align", type=int, default=None)
    p.add_argument("--moe_every", type=int, default=None)
    p.add_argument("--n_experts", type=int, default=None)
    p.add_argument("--moe_top_k", type=int, default=None)
    p.add_argument("--moe_aux_weight", type=float, default=None)
    p.add_argument("--homeo_target", type=float, default=None)
    p.add_argument("--homeo_weight", type=float, default=None)
    p.add_argument("--layerdrop_max", type=float, default=None)
    p.add_argument("--use_confidence_head", type=lambda x: x.lower() == "true", default=None)
    p.add_argument("--confidence_weight", type=float, default=None)
    p.add_argument("--bias", type=lambda x: x.lower() == "true", default=None)
    p.add_argument("--grad_checkpoint", type=lambda x: x.lower() == "true", default=None)


def main():
    parser = argparse.ArgumentParser(description="TensegrityLM — byte-level LM, single-file implementation")
    sub = parser.add_subparsers(dest="command", required=True)

    # prepare
    p_prep = sub.add_parser("prepare", help="Tokenize a text file/directory into train/val .bin files")
    p_prep.add_argument("--data_path", type=str, required=True)
    p_prep.add_argument("--out_dir", type=str, required=True)
    p_prep.add_argument("--val_fraction", type=float, default=0.1)
    p_prep.add_argument("--seed", type=int, default=1337)

    # train
    p_train = sub.add_parser("train", help="Train (or resume) a model")
    p_train.add_argument("--data_dir", type=str, required=True)
    p_train.add_argument("--out_dir", type=str, required=True)
    p_train.add_argument("--init_from", type=str, default="scratch", choices=["scratch", "resume"])
    p_train.add_argument("--device", type=str, default="auto")
    p_train.add_argument("--dtype", type=str, default="auto", choices=["auto", "float32", "bfloat16", "float16"])
    p_train.add_argument("--seed", type=int, default=1337)
    p_train.add_argument("--batch_size", type=int, default=12)
    p_train.add_argument("--grad_accum_steps", type=int, default=1)
    p_train.add_argument("--max_steps", type=int, default=5000)
    p_train.add_argument("--warmup_steps", type=int, default=100)
    p_train.add_argument("--learning_rate", type=float, default=3e-4)
    p_train.add_argument("--min_lr", type=float, default=3e-5)
    p_train.add_argument("--weight_decay", type=float, default=0.1)
    p_train.add_argument("--beta1", type=float, default=0.9)
    p_train.add_argument("--beta2", type=float, default=0.95)
    p_train.add_argument("--grad_clip", type=float, default=1.0)
    p_train.add_argument("--log_interval", type=int, default=10)
    p_train.add_argument("--eval_interval", type=int, default=250)
    p_train.add_argument("--eval_iters", type=int, default=20)
    p_train.add_argument("--compile", action="store_true")
    add_model_args(p_train)

    # generate
    p_gen = sub.add_parser("generate", help="Generate text from a checkpoint")
    p_gen.add_argument("--ckpt", type=str, required=True)
    p_gen.add_argument("--device", type=str, default="auto")
    p_gen.add_argument("--prompt", type=str, default="")
    p_gen.add_argument("--add_bos", action="store_true")
    p_gen.add_argument("--max_new_tokens", type=int, default=200)
    p_gen.add_argument("--temperature", type=float, default=0.8)
    p_gen.add_argument("--top_k", type=int, default=None)
    p_gen.add_argument("--top_p", type=float, default=None)
    p_gen.add_argument("--use_confidence_temp", action="store_true")
    p_gen.add_argument("--no_stop_on_eos", action="store_true")
    p_gen.add_argument("--seed", type=int, default=None)

    # eval
    p_eval = sub.add_parser("eval", help="Evaluate a checkpoint on val data")
    p_eval.add_argument("--ckpt", type=str, required=True)
    p_eval.add_argument("--data_dir", type=str, required=True)
    p_eval.add_argument("--device", type=str, default="auto")
    p_eval.add_argument("--dtype", type=str, default="auto", choices=["auto", "float32", "bfloat16", "float16"])
    p_eval.add_argument("--eval_iters", type=int, default=50)

    args = parser.parse_args()

    if args.command == "prepare":
        prepare_data(args.data_path, args.out_dir, args.val_fraction, args.seed)
    elif args.command == "train":
        cmd_train(args)
    elif args.command == "generate":
        cmd_generate(args)
    elif args.command == "eval":
        cmd_eval(args)
    else:
        parser.print_help()
        sys.exit(1)


if __name__ == "__main__":
    main()


Writing tensegrity_lm.py


## 2. Write `tensegrity_nb_lib.py`

Notebook-specific extensions: replay-based multi-dataset batching, Net2Net-style
model growth (depth + FFN width) with a function-preservation check, Elastic
Weight Consolidation (EWC) for retention, a single-device training loop, and
safetensors export/import.


In [7]:
%%writefile tensegrity_nb_lib.py
"""
tensegrity_nb_lib.py — extension utilities for the TensegrityLM notebook.

These functions are designed to be dropped into notebook cells. They build on
top of tensegrity_lm.py (TensegrityConfig, TensegrityLM, Block, SwiGLU,
MoEFeedForward, ffn_hidden_dim, get_batch, evaluate, save_checkpoint,
load_checkpoint, capture_rng_state, get_lr, pick_dtype).
"""

import os
import json
import dataclasses
import time
import numpy as np
import torch

from tensegrity_lm import (
    TensegrityConfig, TensegrityLM, ByteTokenizer,
    ffn_hidden_dim, get_batch, evaluate, save_checkpoint, load_checkpoint,
    capture_rng_state, get_lr, pick_dtype,
)


# =============================================================================
# Multi-dataset ("replay") batching
# =============================================================================

def get_batch_mixed(data_dirs, weights, split, batch_size, block_size, device):
    """
    Like get_batch, but each row of the batch is sampled from one of several
    prepared data directories, chosen according to `weights` (normalized).
    Used to mix "old" and "new" corpora during continual training so the model
    keeps seeing examples of what it already learned (replay-based retention).
    """
    w = np.asarray(weights, dtype=np.float64)
    w = w / w.sum()

    arrs = []
    for d in data_dirs:
        path = os.path.join(d, f"{split}.bin")
        arr = np.memmap(path, dtype=np.uint16, mode="r")
        if len(arr) <= block_size + 1:
            raise ValueError(
                f"{path} has only {len(arr)} tokens, need > block_size+1 ({block_size + 1})."
            )
        arrs.append(arr)

    xs, ys = [], []
    choices = np.random.choice(len(data_dirs), size=batch_size, p=w)
    for d_idx in choices:
        arr = arrs[d_idx]
        max_start = len(arr) - block_size - 1
        s = np.random.randint(0, max_start)
        xs.append(arr[s:s + block_size].astype(np.int64))
        ys.append(arr[s + 1:s + 1 + block_size].astype(np.int64))

    x_t = torch.from_numpy(np.stack(xs))
    y_t = torch.from_numpy(np.stack(ys))
    if device == "cuda":
        x_t = x_t.pin_memory().to(device, non_blocking=True)
        y_t = y_t.pin_memory().to(device, non_blocking=True)
    else:
        x_t = x_t.to(device)
        y_t = y_t.to(device)
    return x_t, y_t


# =============================================================================
# Net2Net-style "wider FFN" growth (function-preserving at the moment of growth)
# =============================================================================

@torch.no_grad()
def widen_swiglu(old_ffn, new_ffn, old_hidden, new_hidden, bias):
    """
    Copy an old SwiGLU module's weights into a wider new SwiGLU module.

    - w_gate / w_up: old rows [0:old_hidden] are copied; the new_ffn's own
      (already randomly initialized) rows [old_hidden:new_hidden] are left as-is.
    - w_down: old columns [0:old_hidden] are copied; new columns
      [old_hidden:new_hidden] are zeroed.

    Because w_down's new input columns are zero, the new hidden units
    contribute exactly zero to the output regardless of what w_gate/w_up
    compute for them -> the FFN's output is IDENTICAL to the old FFN's output
    for any input, at the moment of growth. The new units only start
    influencing the output once gradients move w_down's new columns away
    from zero during continued training.
    """
    assert new_hidden >= old_hidden, "widen_swiglu only supports growing (new_hidden >= old_hidden)"

    new_ffn.w_gate.weight.data[:old_hidden, :].copy_(old_ffn.w_gate.weight.data)
    new_ffn.w_up.weight.data[:old_hidden, :].copy_(old_ffn.w_up.weight.data)

    new_ffn.w_down.weight.data[:, :old_hidden].copy_(old_ffn.w_down.weight.data)
    new_ffn.w_down.weight.data[:, old_hidden:].zero_()

    if bias:
        new_ffn.w_gate.bias.data[:old_hidden].copy_(old_ffn.w_gate.bias.data)
        new_ffn.w_up.bias.data[:old_hidden].copy_(old_ffn.w_up.bias.data)
        new_ffn.w_down.bias.data.copy_(old_ffn.w_down.bias.data)


@torch.no_grad()
def transplant_block(old_block, new_block, cfg_old, cfg_new):
    """
    Copy everything from old_block into new_block. Attention, norms, and
    residual gains are copied verbatim (their shapes never change under the
    supported growth modes). The FFN is copied verbatim if its hidden_dim is
    unchanged, or widened via widen_swiglu if cfg_new.ffn_mult/ffn_align
    produced a larger hidden_dim.
    """
    # --- Norms ---
    new_block.ln1.weight.data.copy_(old_block.ln1.weight.data)
    new_block.ln2.weight.data.copy_(old_block.ln2.weight.data)

    # --- Attention (shape-invariant under our growth modes) ---
    for name in ["q_proj", "k_proj", "v_proj", "o_proj"]:
        old_lin = getattr(old_block.attn, name)
        new_lin = getattr(new_block.attn, name)
        new_lin.weight.data.copy_(old_lin.weight.data)
        if cfg_old.bias:
            new_lin.bias.data.copy_(old_lin.bias.data)

    # --- Residual gains ---
    new_block.attn_gain.data.copy_(old_block.attn_gain.data)
    new_block.ffn_gain.data.copy_(old_block.ffn_gain.data)

    # --- FFN ---
    old_hidden = ffn_hidden_dim(cfg_old.n_embd, cfg_old.ffn_mult, cfg_old.ffn_align)
    new_hidden = ffn_hidden_dim(cfg_new.n_embd, cfg_new.ffn_mult, cfg_new.ffn_align)

    if old_block.is_moe:
        new_block.ffn.router.weight.data.copy_(old_block.ffn.router.weight.data)
        for old_expert, new_expert in zip(old_block.ffn.experts, new_block.ffn.experts):
            if old_hidden == new_hidden:
                new_expert.w_gate.weight.data.copy_(old_expert.w_gate.weight.data)
                new_expert.w_up.weight.data.copy_(old_expert.w_up.weight.data)
                new_expert.w_down.weight.data.copy_(old_expert.w_down.weight.data)
                if cfg_old.bias:
                    new_expert.w_gate.bias.data.copy_(old_expert.w_gate.bias.data)
                    new_expert.w_up.bias.data.copy_(old_expert.w_up.bias.data)
                    new_expert.w_down.bias.data.copy_(old_expert.w_down.bias.data)
            else:
                widen_swiglu(old_expert, new_expert, old_hidden, new_hidden, cfg_new.bias)
    else:
        if old_hidden == new_hidden:
            new_block.ffn.w_gate.weight.data.copy_(old_block.ffn.w_gate.weight.data)
            new_block.ffn.w_up.weight.data.copy_(old_block.ffn.w_up.weight.data)
            new_block.ffn.w_down.weight.data.copy_(old_block.ffn.w_down.weight.data)
            if cfg_old.bias:
                new_block.ffn.w_gate.bias.data.copy_(old_block.ffn.w_gate.bias.data)
                new_block.ffn.w_up.bias.data.copy_(old_block.ffn.w_up.bias.data)
                new_block.ffn.w_down.bias.data.copy_(old_block.ffn.w_down.bias.data)
        else:
            widen_swiglu(old_block.ffn, new_block.ffn, old_hidden, new_hidden, cfg_new.bias)


_GROWTH_LOCKED_FIELDS = [
    "vocab_size", "block_size", "n_embd", "n_head", "n_kv_head",
    "sliding_window", "global_every", "moe_every", "n_experts", "moe_top_k",
    "bias", "dropout", "init_std", "homeo_target", "homeo_weight",
    "use_confidence_head", "confidence_weight", "moe_aux_weight",
]


@torch.no_grad()
def grow_model(old_model, old_cfg, new_n_layer=None, ffn_growth_mult=None, device=None):
    """
    Create a strictly more capable model that, at the moment of growth,
    computes EXACTLY the same function as old_model (verified by the equality
    check in the next cell).

    Two independent growth axes are supported, and either or both may be used:

      - new_n_layer:    append (new_n_layer - old_cfg.n_layer) new transformer
                        blocks at the end of the stack. New blocks have their
                        residual gains initialized to exactly 0, so they are
                        a no-op at init: x = x + 0*attn_out = x.
      - ffn_growth_mult: widen every block's FFN hidden dimension by this
                        factor (Net2Net "wider net"). New hidden units
                        contribute exactly 0 to the residual stream at init
                        because w_down's new input columns are zeroed.

    n_embd, n_head, n_kv_head, vocab_size, block_size, sliding_window,
    global_every, moe_every, n_experts, moe_top_k, bias, and the
    homeostasis/confidence/MoE-aux hyperparameters are NOT changed by this
    function (changing them would not be exactly function-preserving without
    additional surgery this notebook does not implement).
    """
    if device is None:
        device = next(old_model.parameters()).device

    new_cfg_kwargs = dataclasses.asdict(old_cfg)

    if new_n_layer is not None:
        assert new_n_layer >= old_cfg.n_layer, "new_n_layer must be >= old_cfg.n_layer"
        new_cfg_kwargs["n_layer"] = new_n_layer
        if new_n_layer != old_cfg.n_layer:
            # let TensegrityConfig.__post_init__ recompute gain_init for the new depth;
            # this only affects the *new* blocks (old blocks' gains are copied below).
            new_cfg_kwargs["gain_init"] = None

    if ffn_growth_mult is not None:
        assert ffn_growth_mult >= 1.0, "ffn_growth_mult must be >= 1.0"
        new_cfg_kwargs["ffn_mult"] = old_cfg.ffn_mult * ffn_growth_mult

    new_cfg = TensegrityConfig(**new_cfg_kwargs)

    # Sanity: confirm none of the "locked" fields drifted.
    for field_name in _GROWTH_LOCKED_FIELDS:
        old_val = getattr(old_cfg, field_name)
        new_val = getattr(new_cfg, field_name)
        assert old_val == new_val, (
            f"grow_model must not change '{field_name}' "
            f"(old={old_val!r}, new={new_val!r}); this would break function preservation."
        )

    old_hidden = ffn_hidden_dim(old_cfg.n_embd, old_cfg.ffn_mult, old_cfg.ffn_align)
    new_hidden = ffn_hidden_dim(new_cfg.n_embd, new_cfg.ffn_mult, new_cfg.ffn_align)
    if new_hidden < old_hidden:
        raise ValueError(
            f"Resulting ffn hidden_dim ({new_hidden}) is smaller than the current one "
            f"({old_hidden}). Increase ffn_growth_mult or leave it as None."
        )

    new_model = TensegrityLM(new_cfg)
    new_model.to(device)
    old_model.to(device)

    # --- Global / shared modules ---
    new_model.token_emb.weight.data.copy_(old_model.token_emb.weight.data)  # also lm_head (tied)
    new_model.ln_f.weight.data.copy_(old_model.ln_f.weight.data)
    if old_cfg.use_confidence_head and new_cfg.use_confidence_head:
        new_model.confidence_head.weight.data.copy_(old_model.confidence_head.weight.data)
        new_model.confidence_head.bias.data.copy_(old_model.confidence_head.bias.data)

    # --- Existing blocks: copy / widen ---
    for i in range(old_cfg.n_layer):
        transplant_block(old_model.blocks[i], new_model.blocks[i], old_cfg, new_cfg)

    # --- New blocks (depth growth only): zero residual gains -> exact no-op ---
    for i in range(old_cfg.n_layer, new_cfg.n_layer):
        new_model.blocks[i].attn_gain.data.zero_()
        new_model.blocks[i].ffn_gain.data.zero_()

    return new_model, new_cfg


@torch.no_grad()
def check_function_preserved(old_model, new_model, cfg, device, n_tokens=48, batch=2, atol=1e-3):
    """
    Feed identical random byte sequences through old_model and new_model
    (both in eval mode, fp32, no autocast) and confirm the logits match.
    This is the empirical proof that grow_model() retained everything the
    old model knew.
    """
    old_model.eval()
    new_model.eval()
    torch.manual_seed(0)
    x = torch.randint(0, cfg.vocab_size, (batch, min(n_tokens, cfg.block_size)), device=device)
    old_logits, _, _ = old_model(x, targets=None)
    new_logits, _, _ = new_model(x, targets=None)
    max_diff = (old_logits - new_logits).abs().max().item()
    ok = max_diff < atol
    print(f"[grow_model] max |old_logits - new_logits| = {max_diff:.3e} "
          f"({'PASS' if ok else 'FAIL'}, atol={atol:.1e})")
    if not ok:
        raise AssertionError(
            "grow_model() did not preserve the model's function. "
            "Do not continue training on this model until this is fixed."
        )
    return max_diff


# =============================================================================
# Elastic Weight Consolidation (EWC) — "retain information"
# =============================================================================

@torch.no_grad()
def _autocast_ctx(device_type, amp_dtype, use_amp):
    return torch.amp.autocast(
        device_type=device_type if device_type != "mps" else "cpu",
        dtype=amp_dtype, enabled=use_amp,
    )


def compute_fisher_diagonal(model, cfg, data_dir, device, device_type, dtype="auto",
                             n_batches=50, batch_size=8):
    """
    Estimate the diagonal of the Fisher information matrix for every
    parameter, using the cross-entropy loss on `n_batches` random batches
    drawn from data_dir's train split. Larger values mean "this weight
    matters more for this dataset; don't let it move much."

    Returns a dict {param_name: fisher_tensor} (same shapes as the model's
    current parameters) and a matching snapshot {param_name: value_tensor}
    of the parameters at the time of computation.
    """
    amp_dtype = pick_dtype(dtype, device_type)
    use_amp = amp_dtype in (torch.float16, torch.bfloat16) and device_type in ("cuda", "cpu")

    was_training = model.training
    model.eval()

    fisher = {n: torch.zeros_like(p, dtype=torch.float32) for n, p in model.named_parameters()
              if p.requires_grad}
    old_params = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}

    for _ in range(n_batches):
        x, y = get_batch(data_dir, "train", batch_size, cfg.block_size, device)
        model.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device_type if device_type != "mps" else "cpu",
                                  dtype=amp_dtype, enabled=use_amp):
            _, losses, _ = model(x, y)
            loss = losses["ce_loss"]
        loss.backward()
        for n, p in model.named_parameters():
            if p.grad is not None:
                fisher[n] += p.grad.detach().float().pow(2)

    for n in fisher:
        fisher[n] /= float(n_batches)

    model.zero_grad(set_to_none=True)
    if was_training:
        model.train()

    return fisher, old_params


@torch.no_grad()
def expand_reference_state(old_fisher, old_params, new_model):
    """
    Re-map an (old_fisher, old_params) pair computed on the pre-growth model
    onto the post-growth model's parameter shapes:

      - Parameters whose shape is unchanged (almost everything: attention,
        norms, gains, embeddings, old FFN columns) are carried over directly.
      - Widened FFN tensors are zero-padded: the old region keeps its real
        Fisher value (and its old value, so EWC pulls it back there), while
        the newly-added region gets Fisher = 0 and old_value = its current
        (freshly initialized / post-grow) value, so EWC imposes NO penalty
        on the new capacity — it is free to specialize on new data.
      - Parameters that did not exist before growth (new transformer blocks)
        are absent from the returned dicts entirely -> also unconstrained.
    """
    new_fisher = {}
    new_old_params = {}
    for name, p in new_model.named_parameters():
        if name not in old_fisher:
            continue
        of = old_fisher[name]
        op = old_params[name]
        if of.shape == p.shape:
            new_fisher[name] = of.to(p.device)
            new_old_params[name] = op.to(p.device)
        else:
            nf = torch.zeros_like(p, dtype=torch.float32)
            nop = p.detach().clone()
            slices = tuple(slice(0, s) for s in of.shape)
            nf[slices] = of.to(p.device)
            nop[slices] = op.to(p.device)
            new_fisher[name] = nf
            new_old_params[name] = nop
    return new_fisher, new_old_params


def ewc_penalty(model, fisher, old_params):
    """sum_i fisher_i * (theta_i - theta_old_i)^2, over parameters present in `fisher`."""
    total = None
    for name, p in model.named_parameters():
        f = fisher.get(name)
        if f is None:
            continue
        term = (f * (p - old_params[name]).pow(2)).sum()
        total = term if total is None else total + term
    if total is None:
        return torch.zeros((), device=next(model.parameters()).device)
    return total


# =============================================================================
# Training loop (notebook-friendly: single device, no DDP/compile)
# =============================================================================

def run_training(model, optimizer, cfg, out_dir, device,
                  data_dirs, data_weights=None,
                  max_steps=1000, warmup_steps=50,
                  learning_rate=3e-4, min_lr=3e-5,
                  grad_clip=1.0, batch_size=12, grad_accum_steps=1,
                  log_interval=10, eval_interval=100, eval_iters=20,
                  dtype="auto", start_step=0, best_val_loss=float("inf"),
                  ewc_fisher=None, ewc_old_params=None, ewc_lambda=0.0):
    """
    Train `model` for steps [start_step, max_steps). Checkpoints
    (ckpt_latest.pt / ckpt_best.pt) and a train_log.jsonl are written to
    out_dir. Validation is always measured against data_dirs[0].

    data_dirs: str (single prepared corpus) or list of prepared corpus dirs.
    data_weights: sampling weights over data_dirs (only used if len > 1);
                   normalized internally. Use this to "replay" an old corpus
                   while training on a new one.
    ewc_*: outputs of compute_fisher_diagonal / expand_reference_state, plus
           a weight ewc_lambda. Set ewc_lambda=0.0 (default) to disable.

    Returns (final_step, best_val_loss).
    """
    os.makedirs(out_dir, exist_ok=True)
    device_type = "cuda" if "cuda" in device else ("mps" if device == "mps" else "cpu")
    amp_dtype = pick_dtype(dtype, device_type)
    use_amp = amp_dtype in (torch.float16, torch.bfloat16) and device_type in ("cuda", "cpu")
    scaler = torch.amp.GradScaler(enabled=(amp_dtype == torch.float16 and device_type == "cuda"))

    if isinstance(data_dirs, str):
        data_dirs = [data_dirs]
    if data_weights is None:
        data_weights = [1.0] * len(data_dirs)
    assert len(data_weights) == len(data_dirs)

    log_path = os.path.join(out_dir, "train_log.jsonl")
    log_file = open(log_path, "a")

    model.train()
    optimizer.zero_grad(set_to_none=True)
    step = start_step
    t0 = time.time()
    last_losses = None

    while step < max_steps:
        lr = get_lr(step, warmup_steps, max_steps, learning_rate, min_lr)
        for g in optimizer.param_groups:
            g["lr"] = lr

        accum_total = 0.0
        for _ in range(grad_accum_steps):
            if len(data_dirs) == 1:
                x, y = get_batch(data_dirs[0], "train", batch_size, cfg.block_size, device)
            else:
                x, y = get_batch_mixed(data_dirs, data_weights, "train", batch_size, cfg.block_size, device)

            with torch.amp.autocast(device_type=device_type if device_type != "mps" else "cpu",
                                      dtype=amp_dtype, enabled=use_amp):
                logits, losses, _ = model(x, y)
                total = losses["total_loss"]
                if ewc_lambda > 0.0 and ewc_fisher is not None:
                    total = total + ewc_lambda * ewc_penalty(model, ewc_fisher, ewc_old_params)
                scaled = total / grad_accum_steps

            if scaler.is_enabled():
                scaler.scale(scaled).backward()
            else:
                scaled.backward()

            accum_total += total.item() / grad_accum_steps
            last_losses = losses

        if grad_clip > 0.0:
            if scaler.is_enabled():
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        if scaler.is_enabled():
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        if step % log_interval == 0:
            dt = time.time() - t0
            ce = last_losses["ce_loss"].item()
            homeo = last_losses["homeo_loss"].item()
            moe_aux = last_losses["moe_aux_loss"].item()
            conf = last_losses.get("confidence_loss")
            conf_str = f" conf={conf.item():.4f}" if conf is not None else ""
            print(f"step {step:6d} | lr {lr:.2e} | total {accum_total:.4f} "
                  f"(ce {ce:.4f} homeo {homeo:.4f} moe {moe_aux:.4f}{conf_str}) | {dt:.1f}s")
            log_file.write(json.dumps({
                "step": step, "lr": lr, "total_loss": accum_total,
                "ce_loss": ce, "homeo_loss": homeo, "moe_aux_loss": moe_aux, "time": dt,
            }) + "\n")
            log_file.flush()

        if step > 0 and step % eval_interval == 0:
            val_loss = evaluate(model, data_dirs[0], cfg, device, device_type, amp_dtype, use_amp, eval_iters)
            print(f"step {step:6d} | val_loss[{os.path.basename(data_dirs[0])}] {val_loss:.4f}")
            log_file.write(json.dumps({"step": step, "val_loss": val_loss, "val_dir": data_dirs[0]}) + "\n")
            log_file.flush()

            rng_state = capture_rng_state()
            save_checkpoint(os.path.join(out_dir, "ckpt_latest.pt"), model, optimizer, cfg, step, best_val_loss, rng_state)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                save_checkpoint(os.path.join(out_dir, "ckpt_best.pt"), model, optimizer, cfg, step, best_val_loss, rng_state)
                print(f"step {step:6d} | new best val_loss {best_val_loss:.4f} -> ckpt_best.pt")

        step += 1

    val_loss = evaluate(model, data_dirs[0], cfg, device, device_type, amp_dtype, use_amp, eval_iters)
    rng_state = capture_rng_state()
    save_checkpoint(os.path.join(out_dir, "ckpt_latest.pt"), model, optimizer, cfg, step, best_val_loss, rng_state)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(os.path.join(out_dir, "ckpt_best.pt"), model, optimizer, cfg, step, best_val_loss, rng_state)
    print(f"[train] done. step={step} final_val_loss[{os.path.basename(data_dirs[0])}]={val_loss:.4f} "
          f"best_val_loss={best_val_loss:.4f}")
    log_file.close()
    return step, best_val_loss


# =============================================================================
# safetensors export / import
# =============================================================================

def export_to_safetensors(model, cfg, out_dir, name="model"):
    """
    Write {out_dir}/{name}.safetensors (weights only) and
    {out_dir}/{name}.config.json (TensegrityConfig, needed to reconstruct
    the model before loading the weights back in).

    token_emb.weight and lm_head.weight are tied (the same tensor); since
    safetensors cannot store two tensors that share memory, lm_head.weight
    is cloned into an independent copy before saving. Both keys are written
    (with identical values), so the file loads cleanly with either a tied or
    untied model.
    """
    from safetensors.torch import save_file

    os.makedirs(out_dir, exist_ok=True)
    state_dict = model.state_dict()

    tensors = {}
    for k, v in state_dict.items():
        t = v.detach().cpu().contiguous()
        tensors[k] = t

    # Break the tie: clone lm_head.weight so it doesn't alias token_emb.weight.
    if "lm_head.weight" in tensors and "token_emb.weight" in tensors:
        if tensors["lm_head.weight"].data_ptr() == tensors["token_emb.weight"].data_ptr():
            tensors["lm_head.weight"] = tensors["lm_head.weight"].clone()

    weights_path = os.path.join(out_dir, f"{name}.safetensors")
    save_file(tensors, weights_path, metadata={"format": "pt"})

    config_path = os.path.join(out_dir, f"{name}.config.json")
    with open(config_path, "w") as f:
        json.dump(dataclasses.asdict(cfg), f, indent=2)

    print(f"[export] wrote {weights_path}")
    print(f"[export] wrote {config_path}")
    return weights_path, config_path


def load_from_safetensors(weights_path, config_path, device="cpu"):
    """
    Reconstruct a TensegrityLM from a .safetensors weights file and its
    companion config.json (as written by export_to_safetensors).
    """
    from safetensors.torch import load_file

    with open(config_path) as f:
        cfg = TensegrityConfig(**json.load(f))

    model = TensegrityLM(cfg)
    state_dict = load_file(weights_path)
    model.load_state_dict(state_dict)
    model.to(device)
    return model, cfg


Writing tensegrity_nb_lib.py


## 3. Imports, device, and working directories

In [8]:
import os
import sys
import time

import numpy as np
import torch

sys.path.insert(0, os.getcwd())

from tensegrity_lm import (
    TensegrityConfig, TensegrityLM, ByteTokenizer,
    prepare_data, get_batch, evaluate,
    save_checkpoint, load_checkpoint, capture_rng_state,
    get_lr, pick_device, pick_dtype,
)
from tensegrity_nb_lib import (
    get_batch_mixed,
    grow_model, check_function_preserved,
    compute_fisher_diagonal, expand_reference_state, ewc_penalty,
    run_training, export_to_safetensors, load_from_safetensors,
)

SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = pick_device("auto")
DEVICE_TYPE = "cuda" if "cuda" in DEVICE else ("mps" if DEVICE == "mps" else "cpu")
print(f"device = {DEVICE}  (device_type = {DEVICE_TYPE})")

ROOT = os.getcwd()
DATA_DIR = os.path.join(ROOT, "data")
RUNS_DIR = os.path.join(ROOT, "runs")
SAMPLE_DIR = os.path.join(ROOT, "sample_data")
for _d in (DATA_DIR, RUNS_DIR, SAMPLE_DIR):
    os.makedirs(_d, exist_ok=True)

tok = ByteTokenizer()


device = cpu  (device_type = cpu)


## 4. Data

Set `CUSTOM_DATA_PATH` to a text file or a directory of files containing the
data you want to train on. The model is byte-level, so any UTF-8 text (or
even arbitrary binary data) works.

If `CUSTOM_DATA_PATH` is left as `None`, a small sample corpus is generated
automatically so the rest of the notebook runs out of the box.


In [9]:
# Point this at your own data: either a single text file, or a directory
# (all files inside it, including subdirectories, will be concatenated).
CUSTOM_DATA_PATH = "/content/data"

FALLBACK_CORPUS_INITIAL = """You wake to the sound of the generator changing pitch. It does this twice a \
day, the same two times, and you have stopped checking the clock. The amber \
bulb over the hatch holds steady -- it has held steady since before you \
arrived, and it will hold steady after you leave, however that happens.

The corridor outside is cold in the way that has settled into the walls, not \
the air. You press a palm to the stone and it does not warm. It never warms. \
Past the third junction, the floor changes from poured concrete to something \
older, fitted stone laid by hands long gone, and the air changes with it -- \
drier, quieter, the kind of quiet that has weight.

There is a door at the end with no handle on this side. You have tried it \
before. You will try it again, because trying is the only motion left that \
still feels like yours.

Somewhere below, the machinery turns over, settles, and waits.

"""

if CUSTOM_DATA_PATH is None:
    sample_path = os.path.join(SAMPLE_DIR, "corpus_initial.txt")
    with open(sample_path, "w", encoding="utf-8") as f:
        f.write(FALLBACK_CORPUS_INITIAL * 8)
    CUSTOM_DATA_PATH = sample_path
    print(f"CUSTOM_DATA_PATH not set -> generated sample corpus at {sample_path}")
else:
    print(f"using CUSTOM_DATA_PATH = {CUSTOM_DATA_PATH}")

DATA_INITIAL = os.path.join(DATA_DIR, "initial")
prepare_data(CUSTOM_DATA_PATH, DATA_INITIAL, val_fraction=0.1, seed=SEED)


using CUSTOM_DATA_PATH = /content/data
[prepare] wrote 4,740,979 train tokens -> /content/data/initial/train.bin
[prepare] wrote 526,775 val tokens -> /content/data/initial/val.bin


## 5. Model configuration

A small configuration that trains quickly on CPU. Every field here can be
changed freely for a from-scratch run. **After training**, the fields listed
in `_GROWTH_LOCKED_FIELDS` (see `tensegrity_nb_lib.py`) — `vocab_size`,
`block_size`, `n_embd`, `n_head`, `n_kv_head`, `sliding_window`,
`global_every`, `moe_every`, `n_experts`, `moe_top_k`, `bias`, `dropout`,
`init_std`, the homeostasis/confidence/MoE-aux settings — are fixed for the
lifetime of this model and can only be changed by training a new model from
scratch. `n_layer` (depth) and `ffn_mult` (FFN width) can both be increased
later via `grow_model()` without losing what the model has learned.


In [10]:
cfg = TensegrityConfig(
    # Core sizes
    n_layer=4,
    n_embd=64,
    n_head=4,
    n_kv_head=2,
    block_size=128,
    dropout=0.0,

    # Attention pattern
    sliding_window=32,
    global_every=3,

    # Feed-forward / MoE
    ffn_mult=8.0 / 3.0,
    ffn_align=32,
    moe_every=2,
    n_experts=2,
    moe_top_k=1,
    moe_aux_weight=0.01,

    # Homeostatic residual-gain regularization
    homeo_target=1.0,
    homeo_weight=0.01,

    # Stochastic depth
    layerdrop_max=0.1,

    # Confidence head
    use_confidence_head=True,
    confidence_weight=0.05,

    # Misc
    bias=False,
    init_std=0.02,
)

model = TensegrityLM(cfg)
model.to(DEVICE)

optimizer = model.configure_optimizer(
    weight_decay=0.1, learning_rate=3e-3, betas=(0.9, 0.95), device_type=DEVICE_TYPE,
)

print(f"model has {sum(p.numel() for p in model.parameters()):,} parameters")


[TensegrityLM] layers=4 (moe_layers=2) n_embd=64 n_head=4 n_kv_head=2 vocab=259 block_size=128
[TensegrityLM] total parameters: 287,817 (0.29M)
model has 287,817 parameters


## 6. Train from scratch (with checkpointing)

`run_training` writes `ckpt_latest.pt` and `ckpt_best.pt` (whichever has the
lowest validation loss so far) to `out_dir`, along with a `train_log.jsonl`
of per-step losses and validation results.

Increase `max_steps` (and shrink `eval_interval`/`log_interval` proportionally
if you like) for a more thorough run; the small numbers below are chosen so
this finishes in well under a minute on CPU with the default configuration.


In [11]:
RUN_INITIAL = os.path.join(RUNS_DIR, "initial")

step, best_val = run_training(
    model, optimizer, cfg, RUN_INITIAL, DEVICE,
    data_dirs=[DATA_INITIAL],
    max_steps=150, warmup_steps=15,
    learning_rate=3e-3, min_lr=3e-4,
    batch_size=16, log_interval=25, eval_interval=50, eval_iters=10,
)
print(f"finished initial training at step={step}, best_val_loss={best_val:.4f}")


step      0 | lr 2.00e-04 | total 5.6048 (ce 5.5490 homeo 0.9880 moe 0.0100 conf=0.7186) | 0.4s
step     25 | lr 2.96e-03 | total 3.5192 (ce 3.4792 homeo 0.7776 moe 0.0102 conf=0.4394) | 4.8s
step     50 | lr 2.58e-03 | total 3.0450 (ce 3.0057 homeo 0.5547 moe 0.0050 conf=0.5754) | 8.5s
step     50 | val_loss[initial] 2.9540
step     50 | new best val_loss 2.9540 -> ckpt_best.pt
step     75 | lr 1.88e-03 | total 2.5519 (ce 2.5053 homeo 0.6874 moe 0.0100 conf=0.5930) | 13.4s
step    100 | lr 1.12e-03 | total 2.4084 (ce 2.3619 homeo 0.6892 moe 0.0100 conf=0.5912) | 16.9s
step    100 | val_loss[initial] 2.3463
step    100 | new best val_loss 2.3463 -> ckpt_best.pt
step    125 | lr 5.22e-04 | total 2.3257 (ce 2.2807 homeo 0.6858 moe 0.0100 conf=0.5639) | 20.8s
[train] done. step=150 final_val_loss[initial]=2.2179 best_val_loss=2.2179
finished initial training at step=150, best_val_loss=2.2179


## 7. Load a checkpoint and generate

Reload the model from `ckpt_best.pt` exactly as you would after restarting
the kernel — reconstructing the config and model from the checkpoint's saved
state, then sampling some text.

**Resuming training on the same architecture** (rather than growing it, as
in the next section) is just as simple: rebuild `model`/`optimizer` from the
checkpoint as below, then also call
`optimizer.load_state_dict(ckpt["optimizer_state_dict"])` and
`restore_rng_state(ckpt["rng_state"])`, and pass
`start_step=ckpt["step"] + 1, best_val_loss=ckpt["best_val_loss"]` to
`run_training`.

> With the tiny default config and the tiny sample corpus, 150 steps is
> nowhere near enough to produce coherent English — expect mostly noise with
> occasional recognizable fragments. That's expected at this scale; it's the
> *mechanics* (load -> generate) being demonstrated, not language quality.
> Larger models trained longer on real data will produce much more coherent
> output.


In [13]:
ckpt = load_checkpoint(os.path.join(RUN_INITIAL, "ckpt_best.pt"), DEVICE)
loaded_cfg = TensegrityConfig(**ckpt["config"])

model = TensegrityLM(loaded_cfg)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE)
cfg = loaded_cfg  # keep cfg in sync with the model from here on

print(f"loaded checkpoint from step={ckpt['step']}, best_val_loss={ckpt['best_val_loss']:.4f}")

model.eval()
prompt_ids = [ByteTokenizer.BOS] + tok.encode("You ")
idx = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
with torch.no_grad():
    out = model.generate(idx, max_new_tokens=120, temperature=0.9, top_k=40, use_confidence_temp=True)
print(tok.decode(out[0].tolist()))


[TensegrityLM] layers=4 (moe_layers=2) n_embd=64 n_head=4 n_kv_head=2 vocab=259 block_size=128
[TensegrityLM] total parameters: 287,817 (0.29M)
loaded checkpoint from step=150, best_val_loss=2.2179
You ondes wh/Evov (hebs ) 2amangedy ind tcplvesy [pbanal f  o](https://Buoikiarnikiten ncwbom14_0-83](200)._6"). (housikiolo


## 8. Grow the model (increase capability)

`grow_model` produces a strictly larger model that, **at the moment of
growth**, computes exactly the same function as the original:

- **Depth growth** (`new_n_layer`): new transformer blocks are appended with
  their residual gains initialized to exactly `0`, so each new block starts
  as a no-op (`x = x + 0 * sublayer(x) == x`).
- **FFN width growth** (`ffn_growth_mult`): every block's SwiGLU (dense or
  MoE) hidden dimension is widened. Old rows/columns are copied verbatim; the
  new hidden units' output columns are zero-initialized, so they contribute
  nothing until gradients move them away from zero during continued training.

`check_function_preserved` feeds identical random input through the old and
new models (eval mode, fp32, no autocast) and asserts the logits match to
within `1e-3`. This is the empirical proof that growth did not disturb
anything the model already knew — `grow_model`'s own internal assertions also
guarantee none of the "locked" config fields changed.


In [14]:
NEW_N_LAYER = cfg.n_layer + 2   # add 2 new transformer blocks
FFN_GROWTH_MULT = 2.0           # double every FFN's hidden width

old_model = model
old_cfg = cfg

new_model, new_cfg = grow_model(
    old_model, old_cfg,
    new_n_layer=NEW_N_LAYER,
    ffn_growth_mult=FFN_GROWTH_MULT,
    device=DEVICE,
)

old_params = sum(p.numel() for p in old_model.parameters())
new_params = sum(p.numel() for p in new_model.parameters())
print(f"parameters : {old_params:,} -> {new_params:,}  (+{new_params - old_params:,})")
print(f"n_layer    : {old_cfg.n_layer} -> {new_cfg.n_layer}")
print(f"ffn_mult   : {old_cfg.ffn_mult:.4f} -> {new_cfg.ffn_mult:.4f}")

max_diff = check_function_preserved(old_model, new_model, old_cfg, DEVICE)


[TensegrityLM] layers=6 (moe_layers=3) n_embd=64 n_head=4 n_kv_head=2 vocab=259 block_size=128
[TensegrityLM] total parameters: 699,853 (0.70M)
parameters : 287,817 -> 699,853  (+412,036)
n_layer    : 4 -> 6
ffn_mult   : 2.6667 -> 5.3333
[grow_model] max |old_logits - new_logits| = 2.146e-06 (PASS, atol=1.0e-03)


## 9. New data for continual learning

Set `NEW_DATA_PATH` to the new data you want the model to learn, alongside
everything it already knows. As with `CUSTOM_DATA_PATH` above, if it's left
as `None` a small sample corpus is generated automatically.


In [15]:
NEW_DATA_PATH = "/content/data2"

FALLBACK_CORPUS_NEW = """The new wing was sealed before you arrived, and someone has only just broken \
the seal. The air inside is different -- warmer, faintly metallic, the smell \
of machinery that has been running without anyone to hear it. Cables run \
along the ceiling in neat unbroken lines, newer than anything else in the \
keep, disappearing into a junction box that hums at a pitch just below \
hearing.

The amber light here is brighter, almost white at the edges, and it does not \
flicker the way the old lights do. Someone built this recently. Someone is \
still building it.

You follow the cables to a room with no windows and a single chair facing a \
wall of dark glass. The glass does not show your reflection. It shows \
nothing, which is its own kind of answer.

A second door, unmarked, stands ajar at the far end -- the first open door \
you have found in this place.

"""

if NEW_DATA_PATH is None:
    sample_path_new = os.path.join(SAMPLE_DIR, "corpus_new.txt")
    with open(sample_path_new, "w", encoding="utf-8") as f:
        f.write(FALLBACK_CORPUS_NEW * 8)
    NEW_DATA_PATH = sample_path_new
    print(f"NEW_DATA_PATH not set -> generated sample corpus at {sample_path_new}")
else:
    print(f"using NEW_DATA_PATH = {NEW_DATA_PATH}")

DATA_NEW = os.path.join(DATA_DIR, "new")
prepare_data(NEW_DATA_PATH, DATA_NEW, val_fraction=0.1, seed=SEED)


using NEW_DATA_PATH = /content/data2
[prepare] wrote 1,983,789 train tokens -> /content/data/new/train.bin
[prepare] wrote 220,421 val tokens -> /content/data/new/val.bin


## 10. Estimate what matters (Fisher information)

`compute_fisher_diagonal` estimates, for every parameter of the **old**
model, how sensitive the training loss on the **old** data is to that
parameter (the diagonal of the Fisher information matrix) and snapshots the
parameter's current value. `expand_reference_state` then remaps both onto the
grown model's (possibly larger) parameter shapes:

- Parameters whose shape didn't change carry their Fisher value and old value
  over directly.
- The old region of a widened FFN tensor keeps its real Fisher value (so EWC
  will resist moving it); the newly-added region gets Fisher `= 0` (so it is
  completely free to specialize on new data).
- Brand-new transformer blocks are entirely absent from these dicts, so they
  too are unconstrained.


In [16]:
print("computing Fisher information on the pre-growth model + initial data...")
fisher, ref_params = compute_fisher_diagonal(
    old_model, old_cfg, DATA_INITIAL, DEVICE, DEVICE_TYPE,
    n_batches=30, batch_size=8,
)
new_fisher, new_ref_params = expand_reference_state(fisher, ref_params, new_model)
print(f"Fisher computed for {len(fisher)} tensors, remapped to {len(new_fisher)} tensors for the grown model")


computing Fisher information on the pre-growth model + initial data...
Fisher computed for 56 tensors, remapped to 56 tensors for the grown model


## 11. Retention baseline (before continual training)

Because growth is function-preserving, the grown model's validation loss on
the original data should match the pre-growth model's. This is our "before"
baseline for the retention check after continual training.


In [17]:
amp_dtype = pick_dtype("auto", DEVICE_TYPE)
use_amp = amp_dtype in (torch.float16, torch.bfloat16) and DEVICE_TYPE in ("cuda", "cpu")

val_initial_before = evaluate(new_model, DATA_INITIAL, new_cfg, DEVICE, DEVICE_TYPE, amp_dtype, use_amp, eval_iters=20)
val_new_before = evaluate(new_model, DATA_NEW, new_cfg, DEVICE, DEVICE_TYPE, amp_dtype, use_amp, eval_iters=20)
print(f"before continual training: val_loss(initial)={val_initial_before:.4f}  val_loss(new)={val_new_before:.4f}")


before continual training: val_loss(initial)=2.1960  val_loss(new)=2.2092


## 12. Continual training (retain old, learn new)

The grown model is trained on a mixture of the old and new data
(`data_weights` controls the sampling ratio — this is the "replay" component
of retention), with an EWC penalty added to the loss that discourages the
parameters covered by `new_fisher` from drifting away from `new_ref_params`.
`ewc_lambda` controls how strongly old knowledge is protected; `0.0` disables
EWC entirely.

A fresh optimizer is created because the grown model has new/resized
parameters that didn't exist in the old optimizer's state.


In [18]:
new_optimizer = new_model.configure_optimizer(
    weight_decay=0.1, learning_rate=3e-3, betas=(0.9, 0.95), device_type=DEVICE_TYPE,
)

RUN_GROWN = os.path.join(RUNS_DIR, "grown")

step2, best_val2 = run_training(
    new_model, new_optimizer, new_cfg, RUN_GROWN, DEVICE,
    data_dirs=[DATA_INITIAL, DATA_NEW], data_weights=[0.5, 0.5],
    max_steps=150, warmup_steps=15,
    learning_rate=3e-3, min_lr=3e-4,
    batch_size=16, log_interval=25, eval_interval=50, eval_iters=10,
    ewc_fisher=new_fisher, ewc_old_params=new_ref_params, ewc_lambda=25.0,
)
print(f"finished continual training at step={step2}, best_val_loss(initial)={best_val2:.4f}")


step      0 | lr 2.00e-04 | total 2.2842 (ce 2.2385 homeo 0.7882 moe 0.0100 conf=0.5567) | 0.3s
step     25 | lr 2.96e-03 | total 2.1390 (ce 2.0956 homeo 0.4431 moe 0.0100 conf=0.5556) | 7.4s
step     50 | lr 2.58e-03 | total 1.8952 (ce 1.8536 homeo 0.4084 moe 0.0100 conf=0.4882) | 14.7s
step     50 | val_loss[initial] 1.8843
step     50 | new best val_loss 1.8843 -> ckpt_best.pt
step     75 | lr 1.88e-03 | total 1.8686 (ce 1.8286 homeo 0.3897 moe 0.0067 conf=0.4960) | 22.8s
step    100 | lr 1.12e-03 | total 1.8031 (ce 1.7623 homeo 0.3838 moe 0.0067 conf=0.4925) | 29.6s
step    100 | val_loss[initial] 1.7077
step    100 | new best val_loss 1.7077 -> ckpt_best.pt
step    125 | lr 5.22e-04 | total 1.7161 (ce 1.6718 homeo 0.3867 moe 0.0100 conf=0.4865) | 37.8s
[train] done. step=150 final_val_loss[initial]=1.7517 best_val_loss=1.7077
finished continual training at step=150, best_val_loss(initial)=1.7077


## 13. Retention check (after continual training)

A successful retention run should show `val_loss(new)` improving (the model
learned the new data) while `val_loss(initial)` does not get substantially
worse — and, with replay, it often improves too.


In [19]:
val_initial_after = evaluate(new_model, DATA_INITIAL, new_cfg, DEVICE, DEVICE_TYPE, amp_dtype, use_amp, eval_iters=20)
val_new_after = evaluate(new_model, DATA_NEW, new_cfg, DEVICE, DEVICE_TYPE, amp_dtype, use_amp, eval_iters=20)

print(f"{'':18s}{'before':>10s}{'after':>10s}")
print(f"{'val_loss(initial)':18s}{val_initial_before:>10.4f}{val_initial_after:>10.4f}")
print(f"{'val_loss(new)':18s}{val_new_before:>10.4f}{val_new_after:>10.4f}")


                      before     after
val_loss(initial)     2.1960    1.7732
val_loss(new)         2.2092    1.5080


## 14. Export to safetensors

Writes `{name}.safetensors` (portable weights) and `{name}.config.json`
(the `TensegrityConfig` needed to reconstruct the model). Because
`token_emb.weight` and `lm_head.weight` are tied (the same tensor) and
safetensors cannot store two tensors that alias the same memory,
`export_to_safetensors` writes an independent clone of `lm_head.weight`.


In [20]:
EXPORT_DIR = os.path.join(RUNS_DIR, "export")
EXPORT_NAME = "tensegrity"

weights_path, config_path = export_to_safetensors(new_model, new_cfg, EXPORT_DIR, name=EXPORT_NAME)
print(f"weights : {weights_path}")
print(f"config  : {config_path}")


[export] wrote /content/runs/export/tensegrity.safetensors
[export] wrote /content/runs/export/tensegrity.config.json
weights : /content/runs/export/tensegrity.safetensors
config  : /content/runs/export/tensegrity.config.json


## 15. Reload from safetensors and verify

Reconstructs the model from `{name}.config.json` + `{name}.safetensors` and
confirms its logits match the in-memory model exactly, then generates a
sample to confirm everything still works end to end (the same caveat about
output quality at this tiny demo scale applies here too).


In [21]:
reloaded_model, reloaded_cfg = load_from_safetensors(weights_path, config_path, device=DEVICE)
reloaded_model.eval()
new_model.eval()

torch.manual_seed(SEED)
x_check = torch.randint(0, new_cfg.vocab_size, (1, 16), device=DEVICE)
with torch.no_grad():
    logits_a, _, _ = new_model(x_check, targets=None)
    logits_b, _, _ = reloaded_model(x_check, targets=None)
print(f"max logit difference after safetensors roundtrip: {(logits_a - logits_b).abs().max().item():.3e}")

prompt_ids = [ByteTokenizer.BOS] + tok.encode("You ")
idx = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
with torch.no_grad():
    out = reloaded_model.generate(idx, max_new_tokens=120, temperature=0.9, top_k=40, use_confidence_temp=True)
print(tok.decode(out[0].tolist()))


[TensegrityLM] layers=6 (moe_layers=3) n_embd=64 n_head=4 n_kv_head=2 vocab=259 block_size=128
[TensegrityLM] total parameters: 699,853 (0.70M)
max logit difference after safetensors roundtrip: 0.000e+00
You oad hateliong of Pesisvorars-lemplt, Con Huunograg caforlrin, ho Wulary asevermabcesd M. (1– Fomfeag ind _Aurated of 2


## Next steps

**Files produced by this notebook** (under `runs/`):

- `runs/initial/ckpt_latest.pt`, `runs/initial/ckpt_best.pt`,
  `runs/initial/train_log.jsonl` — the from-scratch run.
- `runs/grown/ckpt_latest.pt`, `runs/grown/ckpt_best.pt`,
  `runs/grown/train_log.jsonl` — the grown model after continual training.
- `runs/export/tensegrity.safetensors`, `runs/export/tensegrity.config.json`
  — portable export of the final (grown, continually-trained) model.

**Repeating the grow + retain cycle.** To grow the model again later, treat
`new_model`/`new_cfg` from this run as the new "old" model/config: call
`grow_model` on them, then recompute `compute_fisher_diagonal` +
`expand_reference_state` using `new_model`/`new_cfg` and whichever data
directories you want to protect (e.g. `[DATA_INITIAL, DATA_NEW]` with combined
weights) before the next round of `run_training`. Each cycle's EWC anchors
only the most recent round; if you need long-horizon retention across many
growth cycles, consider periodically re-estimating Fisher information over a
mixture of *all* prior datasets rather than just the most recent one.

**Scaling up.** The configuration in this notebook (`n_layer=4`, `n_embd=64`,
`block_size=128`, ~tens of thousands of parameters) is sized to run quickly on
CPU as a demonstration. For real training:

- Increase `n_embd`, `n_layer`, `n_head`/`n_kv_head`, `block_size`,
  `n_experts`, and `ffn_mult`/`ffn_align` in the config cell.
- Increase `max_steps`, and tune `batch_size`, `warmup_steps`,
  `learning_rate`/`min_lr`, and `eval_interval` in the training cells.
- Use a GPU (`pick_device("auto")` will select CUDA automatically if
  available) — `pick_dtype` will then pick `bfloat16`/`float16` automatically
  for autocast.
- Use a much larger, more diverse text corpus for `CUSTOM_DATA_PATH` /
  `NEW_DATA_PATH` — the sample corpora generated here are deliberately tiny
  and repetitive, useful only for exercising the pipeline mechanics.

**What `grow_model` does *not* do.** `n_embd` (and everything derived from
it: `n_head`, `n_kv_head`, RoPE head dimension, RMSNorm shapes, the residual
stream width) is in `_GROWTH_LOCKED_FIELDS` and cannot be grown by this
notebook — widening the residual stream itself is not cleanly
function-preserving without additional surgery (in particular, RMSNorm's
learned scale would need to be re-derived rather than copied). If you need a
wider residual stream, train a new model from scratch at that width — you can
still use `compute_fisher_diagonal` on the old model/data and
`expand_reference_state` as a *starting point* for an EWC penalty on the
overlapping subset of parameters, but this notebook's `grow_model` will raise
an assertion error if you try to change `n_embd` directly.
